### 1. 

In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("zoomcamp_hw") \
    .getOrCreate()

print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/17 01:51:40 WARN Utils: Your hostname, coorui-M2-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.22 instead (on interface en0)
26/03/17 01:51:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/17 01:51:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.1.1


### 2.

In [2]:
import urllib.request
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Spark Session 시작
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("zoomcamp_hw") \
    .getOrCreate()

print("✅ Q1 정답 (Spark Version):", spark.version)
print("-" * 30)

# --- 데이터 다운로드 ---
print("⏳ 데이터를 다운로드하고 있습니다. 잠시만 기다려주세요...")
yellow_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"
urllib.request.urlretrieve(yellow_url, "yellow_2025_11.parquet")

zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
urllib.request.urlretrieve(zone_url, "taxi_zone_lookup.csv")

# --- DF 로드 ---
df = spark.read.parquet("yellow_2025_11.parquet")
df_zones = spark.read.option("header", "true").csv("taxi_zone_lookup.csv")

✅ Q1 정답 (Spark Version): 4.1.1
------------------------------
⏳ 데이터를 다운로드하고 있습니다. 잠시만 기다려주세요...


In [9]:
# Q2: 파티션 나누고 저장
# 폴더에서 용량 확인해야 함
print("✅ Q2 진행 중: 데이터를 4개의 파티션으로 나누어 저장합니다...")
df.repartition(4).write.parquet("data/pq/yellow/2025/11/", mode="overwrite")
print("👉 Q2 정답 힌트: 'data/pq/yellow/2025/11/' 폴더에 생성된 파일 하나의 용량을 확인해보세요! (보기: 6MB, 25MB, 75MB, 100MB)")

✅ Q2 진행 중: 데이터를 4개의 파티션으로 나누어 저장합니다...


👉 Q2 정답 힌트: 'data/pq/yellow/2025/11/' 폴더에 생성된 파일 하나의 용량을 확인해보세요! (보기: 6MB, 25MB, 75MB, 100MB)


### 3.

In [8]:
# Q3: 15일 레코드 수 카운트 (11월 15일 기준)
q3_count = df.filter(F.to_date(df.tpep_pickup_datetime) == '2025-11-15').count()
print("✅ Q3 정답 (15일 레코드 수):", q3_count)

✅ Q3 정답 (15일 레코드 수): 162604


### 4.

In [10]:
# Q4: 가장 긴 운행 시간 (시간 단위)

# unix_timestamp 함수를 사용해서 안전하게 초 단위로 변환 후 3600으로 나눕니다.
df_duration = df.withColumn('duration_hours', 
                            (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600)

# 가장 큰 값을 찾아서 소수점 첫째 자리까지 반올림
q4_ans = df_duration.select(F.max('duration_hours')).collect()[0][0]
print("✅ Q4 정답 (가장 긴 운행 시간):", round(q4_ans, 1))

✅ Q4 정답 (가장 긴 운행 시간): 90.6


### 5.

- 기본 포트 번호는 4040

### 6.

In [6]:
# Q6: 가장 빈도가 적은 지역(Zone)
print("✅ Q6 정답 (가장 빈도가 적은 픽업 지역):")
df.join(df_zones, df.PULocationID == df_zones.LocationID) \
  .groupBy('Zone') \
  .count() \
  .orderBy('count') \
  .show(1)

✅ Q6 정답 (가장 빈도가 적은 픽업 지역):
+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
+--------------------+-----+
only showing top 1 row
